In [7]:
import os
import pandas as pd
import plotly.graph_objects as go
from typing import List, Optional, Dict
import numpy as np

def find_config_file(folder_path: str) -> Optional[str]:
    """
    Finds a configuration file (ending with .txt) within the 'configs' subfolder.
    """
    config_dir = os.path.join(folder_path, 'configs')
    if not os.path.isdir(config_dir):
        return None
    for item in os.listdir(config_dir):
        if item.endswith('.txt'):
            return os.path.join(config_dir, item)
    return None

def parse_config(file_path: str) -> Dict[str, str]:
    """
    Parses a 'key = value', 'key value', or 'key: value' configuration file into a dictionary.
    """
    params = {}
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                
                separator = None
                if ':' in line:
                    separator = ':'
                elif '=' in line:
                    separator = '='

                if separator:
                    parts = line.split(separator, 1)
                else:
                    parts = line.split(None, 1)

                if len(parts) == 2:
                    key, value = parts
                    params[key.strip().lower()] = value.strip()
    except FileNotFoundError:
        print(f"Config file not found: {file_path}")
    except Exception as e:
        print(f"Error parsing config file {file_path}: {e}")
    return params

def find_timing_file(run_folder_path: str, sim_type: str) -> Optional[str]:
    """Finds the ...trace_matched_timing.csv file for a given run."""
    sim_output_dir = os.path.join(run_folder_path, sim_type.lower())
    if not os.path.isdir(sim_output_dir):
        return None
    for f in os.listdir(sim_output_dir):
        if 'trace_matched_timing.csv' in f:
            return os.path.join(sim_output_dir, f)
    return None

# --- Collective Info Parsing ---
def parse_collectives_log(log_path: str) -> Dict[str, Dict]:
    """Parses a duplicate_collectives.log file to extract signatures."""
    collective_info = {}
    try:
        with open(log_path, 'r') as f:
            lines = f.readlines()
            i = 0
            while i < len(lines):
                line = lines[i]
                workload_match = re.match(r'^Workload: (\S+)', line)
                if workload_match:
                    current_workload = workload_match.group(1)
                    # Look for signature on the next line
                    if (i + 1 < len(lines)) and (signature_match := re.match(r'^\s+Signature: \((.*)\)', lines[i+1])):
                        sig_content = signature_match.group(1).strip()
                        # Split signature into its three parts
                        parts = sig_content.rsplit(', ', 2)
                        if len(parts) == 3:
                            npu_tuples, comm_type, comm_size = parts
                            collective_info[current_workload] = {
                                'npu_tuples': npu_tuples.strip(),
                                'comm_type': comm_type.strip(),
                                'comm_size': comm_size.strip()
                            }
                i += 1
    except FileNotFoundError:
        print(f"Warning: Collectives log file not found at {log_path}")
    except Exception as e:
        print(f"Error parsing collectives log {log_path}: {e}")
    return collective_info

In [8]:
import re
import pandas as pd
from plotly.subplots import make_subplots

# --- Configuration ---
base_comparison_folders = [
    '/app/astra-sim/upc/output/comparison_run/experiment2/Dragonfly/multiple_collectives_deterministic'
]
comparison_plot_metric = 'avg'  # Can be 'avg' or 'max'

# --- Helper Functions ---
cc_modes = {0: "PFC", 1: "DCQCN", 3: "HPCC", 7: "TIMELY", 8: "DCTCP", 10: "HPCC-PINT"}
workload_info_regex = re.compile(r'([a-zA-Z_]+)_size_(\d+)_(\d+)')

def parse_runtime(time_str: str) -> float:
    """Parses 'H:MM:SS.ffffff' into seconds."""
    if not time_str: return 0.0
    try:
        h, m, s = map(float, time_str.split(':'))
        return h * 3600 + m * 60 + s
    except (ValueError, IndexError):
        return 0.0

def process_simulation(folder: str, sim_type: str, summary_params: dict, topo_file: str, run_name: str, workload_name: str, comm_size: int, group_id: int) -> Optional[dict]:
    """Processes a single simulation run, extracts timing data, and returns a result dictionary."""
    timing_file = find_timing_file(folder, sim_type)
    if not timing_file:
        return None

    try:
        df = pd.read_csv(timing_file)
        if sim_type == 'ns3' and 'node_name' in df.columns:
            df = df[df['node_name'] != 'dummy_node'].copy()

        time_col = 'callback_tick'
        if time_col not in df.columns:
            return None

        df = df[df[time_col] > 100].copy()
        elapsed_times = df[time_col].dropna()
        if elapsed_times.empty:
            return None

        workload_name = summary_params.get('collective', 'N/A').strip()

        return {
            'workload': workload_name,
            'comm_size': comm_size,
            'comm_type': workload_name,
            'group_id': group_id,
            'npu_count': summary_params.get('npus count', 'N/A'),
            'topology': ('_'.join(topo_file.split('_')[1:]).rsplit('.', 1)[0]),
            'sim_type': sim_type.upper(),
            'run_name': run_name,
            'avg_time': elapsed_times.mean(),
            'max_time': elapsed_times.max(),
            'min_time': elapsed_times.min(),
            'std_dev': elapsed_times.std(),
            'execution_time': parse_runtime(summary_params.get('total runtime', '0:0:0.0')),
            'path': folder
        }
    except Exception as e:
        print(f"Error processing {sim_type} in {folder}: {e}")
        return None

# --- Data Collection Logic ---
all_run_folders = []
for base_folder in base_comparison_folders:
    for workload_folder in os.listdir(base_folder):
        workload_path = os.path.join(base_folder, workload_folder)
        if os.path.isdir(workload_path):
            all_run_folders.extend([os.path.join(workload_path, d) for d in os.listdir(workload_path) if os.path.isdir(os.path.join(workload_path, d))])

comparison_results = []
for folder in sorted(all_run_folders):
    run_summary_path = os.path.join(folder, 'run_summary.txt')
    if not os.path.exists(run_summary_path):
        continue

    summary_params = parse_config(run_summary_path)
    
    # Extract info from workload folder name
    workload_folder_name = os.path.basename(os.path.dirname(folder))
    workload_name, comm_size, group_id = 'N/A', -1, -1
    workload_match = workload_info_regex.match(workload_folder_name)
    if workload_match:
        workload_name = workload_match.group(1)
        comm_size = int(workload_match.group(2))
        group_id = int(workload_match.group(3))

    # Process all potential simulation types in the folder
    sim_dirs = [d for d in os.listdir(folder) if os.path.isdir(os.path.join(folder, d)) and d in ['analytical_unaware', 'g2', 'ns3']]

    for sim_type in sim_dirs:
        topo_file = "N/A"
        run_name = f"{sim_type.replace('_', ' ').title()}"

        if sim_type != 'analytical_unaware':
            topo_key = 'g2 topology file override' if sim_type == 'g2' else 'ns3 topology file override'
            topo_path = summary_params.get(topo_key, '')
            if 'all_paths' in topo_path: continue
            if not topo_path: continue
            topo_file = os.path.basename(topo_path)

            if sim_type == 'ns3':
                ns3_config_file = find_config_file(folder)
                if ns3_config_file:
                    ns3_params = parse_config(ns3_config_file)
                    run_name = (
                        f"NS3 (cc:{cc_modes.get(int(ns3_params.get('cc_mode', -1)), 'N/A')}, "
                        f"win:{ns3_params.get('has_win', 'N/A')}, adapt:{ns3_params.get('var_win', 'N/A')}, "
                        f"buf:{ns3_params.get('buffer_size', 'N/A')}, size:{ns3_params.get('packet_payload_size', 'N/A')})"
                    )
            else: # G2
                run_name = "G2"

        result = process_simulation(folder, sim_type, summary_params, topo_file, run_name, workload_name, comm_size, group_id)
        if result:
            comparison_results.append(result)

# --- Plotting Logic ---


In [9]:
# --- Analysis and Plotting ---
if comparison_results:
    comp_df = pd.DataFrame(comparison_results)

    # Map for collective communication types
    comm_type_map = {
        "all_reduce": "ALL_REDUCE", "reduce": "REDUCE", "all_gather": "ALL_GATHER", "gather": "GATHER",
        "scatter": "SCATTER", "broadcast": "BROADCAST", "all_to_all": "ALL_TO_ALL", "reduce_scatter": "REDUCE_SCATTER",
        "reduce_scatter_block": "REDUCE_SCATTER_BLOCK", "barrier": "BARRIER"
    }
    comp_df['comm_type_str'] = comp_df['comm_type'].astype(str).map(comm_type_map).fillna(comp_df['comm_type'])

    # --- Divergence Analysis (per topology) ---
    metric_col = 'max_time'
    metric_name = 'Maximum Time'
    
    print(f"--- Analyzing Divergence for: {metric_name} ---")
    
    divergence_data = []
    # Group by workload, group_id, and topology to compare G2 and NS3 for each specific case
    for group_keys, group_df in comp_df.groupby(['workload', 'group_id', 'topology']):
        wl, gid, topo = group_keys
        
        g2_runs = group_df[group_df['sim_type'] == 'G2']
        ns3_runs = group_df[group_df['sim_type'] == 'NS3']

        if g2_runs.empty or ns3_runs.empty:
            continue

        # There should be only one G2 run per group
        g2_run = g2_runs.iloc[0]
        g2_time = g2_run[metric_col]

        # Find the best (minimum time) NS3 run within this group
        best_ns3_run = ns3_runs.loc[ns3_runs[metric_col].idxmin()]
        best_ns3_time = best_ns3_run[metric_col]

        if g2_time < 100 or best_ns3_time < 100:
            continue

        divergence = (best_ns3_time - g2_time) / g2_time
        
        divergence_data.append({
            'Workload': wl,
            'Comm Type': g2_run.get('comm_type_str', 'N/A'),
            'Comm Size': g2_run.get('comm_size', 'N/A'),
            'Group ID': gid,
            'Topo': topo,
            'G2 Time': g2_time,
            'Best NS3 Time': best_ns3_time,
            'Best NS3 Run': best_ns3_run['run_name'],
            'Divergence': divergence,
            'npu_tuples': g2_run.get('npu_tuples', 'N/A'),
            'g2_path': g2_run['path'],
            'ns3_path': best_ns3_run['path']
        })

    if not divergence_data:
        print(f"No divergent workloads found for {metric_name}.\n")
    else:
        div_df = pd.DataFrame(divergence_data).sort_values(by='Divergence', ascending=True)

        # --- Display The New Table ---
        display_df = div_df.copy()
        # Convert times to ms and format divergence
        display_df['G2 Time (ms)'] = display_df['G2 Time'] / 1_000_000
        display_df['Best NS3 Time (ms)'] = display_df['Best NS3 Time'] / 1_000_000
        display_df['Divergence'] = display_df['Divergence'].apply(lambda x: f"{x:+.2%}")

        print(f"\n--- Workload Divergence Analysis ({metric_name}) ---")
        with pd.option_context('display.float_format', '{:,.3f}'.format):
            display(display_df[[
                'Comm Type', 'Comm Size', 'Group ID', 'G2 Time (ms)', 
                'Best NS3 Time (ms)', 'Divergence', 'Topo'
            ]])

        # --- Plotting Logic for Top Divergent Cases ---
        top_10_workloads = div_df.head(10)
        print(f"\n--- Generating Plots for Top 10 Divergent Workloads ({metric_name}) ---")
        for _, row in top_10_workloads.iterrows():
            # Get all runs for this specific workload, group, and topology
            workload_df = comp_df[
                (comp_df['workload'] == row['Workload']) &
                (comp_df['group_id'] == row['Group ID']) &
                (comp_df['topology'] == row['Topo'])
            ]
        
            g2_run = workload_df[workload_df['sim_type'] == 'G2'].iloc[0]
            ns3_runs_for_plot = workload_df[workload_df['sim_type'] == 'NS3'].sort_values(by=metric_col)
            
            fig = make_subplots(rows=1, cols=2, subplot_titles=(f"Performance ({metric_name})", "Per-NPU Time Correlation"))

            # Bar plot
            fig.add_trace(go.Bar(
                x=ns3_runs_for_plot['run_name'], 
                y=ns3_runs_for_plot[metric_col] / 1_000_000, 
                name='NS3 Runs'
            ), row=1, col=1)
            fig.add_hline(
                y=g2_run[metric_col] / 1_000_000, 
                line_dash="dot", 
                annotation_text=f"G2 Time", 
                row=1, col=1
            )
            fig.update_yaxes(title_text="Time (ms)", row=1, col=1)


            # Scatter plot
            g2_timing_file = find_timing_file(row['g2_path'], 'G2')
            ns3_timing_file = find_timing_file(row['ns3_path'], 'NS3')
            if g2_timing_file and ns3_timing_file:
                try:
                    df_g2 = pd.read_csv(g2_timing_file)[['sys_id', 'callback_tick']].rename(columns={'callback_tick': 'g2_time'})
                    df_ns3 = pd.read_csv(ns3_timing_file)[['sys_id', 'callback_tick']].rename(columns={'callback_tick': 'ns3_time'})
                    merged_df = pd.merge(df_ns3, df_g2, on='sys_id')
                    merged_df = merged_df[(merged_df['ns3_time'] >= 100) & (merged_df['g2_time'] >= 100)]
                    
                    merged_df['g2_time_ms'] = merged_df['g2_time'] / 1_000_000
                    merged_df['ns3_time_ms'] = merged_df['ns3_time'] / 1_000_000

                    fig.add_trace(go.Scatter(x=merged_df['ns3_time_ms'], y=merged_df['g2_time_ms'], mode='markers', name='NPU Times'), row=1, col=2)
                    min_val = min(merged_df['ns3_time_ms'].min(), merged_df['g2_time_ms'].min())
                    max_val = max(merged_df['ns3_time_ms'].max(), merged_df['g2_time_ms'].max())
                    fig.add_trace(go.Scatter(x=[min_val, max_val], y=[min_val, max_val], mode='lines', name='y=x'), row=1, col=2)
                    fig.update_xaxes(title_text="NS3 Time (ms)", row=1, col=2)
                    fig.update_yaxes(title_text="G2 Time (ms)", row=1, col=2)
                except Exception as e:
                    print(f"Could not create scatter plot for {row['Workload']}: {e}")
            
            plot_title = (f"<b>{row['Workload']} (Group: {row['Group ID']}, Topo: {row['Topo']})</b><br>"
                          f'Metric: {metric_name}<br>'
                          f"Comm: {row['Comm Type']}, Size: {row['Comm Size']}, NPUs: {row['npu_tuples']}")

            fig.update_layout(title_text=plot_title, height=700, margin=dict(t=150), showlegend=False)
            fig.show()
        print("\n" + "="*80 + "\n")

else:
    print("No comparison results to process.")

--- Analyzing Divergence for: Maximum Time ---

--- Workload Divergence Analysis (Maximum Time) ---


,Comm Type,Comm Size,Group ID,G2 Time (ms),Best NS3 Time (ms),Divergence,Topo
5,multiple_collectives_deterministic/all_gather_...,-1,-1,0.677,0.594,-12.29%,Dragonfly_16_v1_Random
17,multiple_collectives_deterministic/reduce_scat...,-1,-1,0.677,0.597,-11.84%,Dragonfly_16_v1_Random
4,multiple_collectives_deterministic/all_gather_...,-1,-1,0.316,0.296,-6.28%,Dragonfly_16_v1_Random
16,multiple_collectives_deterministic/reduce_scat...,-1,-1,0.317,0.298,-6.11%,Dragonfly_16_v1_Random
11,multiple_collectives_deterministic/all_gather_...,-1,-1,5.420,5.091,-6.06%,Dragonfly_16_v1_Random
23,multiple_collectives_deterministic/reduce_scat...,-1,-1,5.415,5.103,-5.77%,Dragonfly_16_v1_Random
18,multiple_collectives_deterministic/reduce_scat...,-1,-1,5.166,5.202,+0.70%,Dragonfly_16_v1_Random
6,multiple_collectives_deterministic/all_gather_...,-1,-1,5.053,5.090,+0.72%,Dragonfly_16_v1_Random
21,multiple_collectives_deterministic/reduce_scat...,-1,-1,1.033,1.041,+0.74%,Dragonfly_16_v1_Random
9,multiple_collectives_deterministic/all_gather_...,-1,-1,1.011,1.018,+0.75%,Dragonfly_16_v1_Random



--- Generating Plots for Top 10 Divergent Workloads (Maximum Time) ---


In [10]:
comp_df['sim_type'].unique()
comp_df['sim_type'].nunique()
comp_df['sim_type'].value_counts()

sim_type
NS3                   48
ANALYTICAL_UNAWARE    24
G2                    24
Name: count, dtype: int64